In [0]:
from datetime import datetime
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import col, lit, array, when, size, filter as sql_filter


In [0]:

DQ_RESULTS_PATH = "abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_control/dq_results"
DQ_RESULTS_SCHEMA = """
    check_id STRING,
    batch_id STRING,
    layer STRING,
    table_name STRING,
    check_name STRING,
    severity STRING,
    rows_checked LONG,
    rows_failed LONG,
    action_taken STRING,
    checked_at TIMESTAMP
"""

In [0]:

def _ensure_dq_results_table_exists(spark: SparkSession) -> None:
    try:
        spark.read.format("delta").load(DQ_RESULTS_PATH)
    except Exception:
        empty_df = spark.createDataFrame([], schema=DQ_RESULTS_SCHEMA)
        empty_df.write.format("delta").mode("overwrite").save(DQ_RESULTS_PATH)

In [0]:

def log_dq_result(
    spark: SparkSession, batch_id: str, layer: str, table_name: str,
    check_name: str, severity: str, rows_checked: int, rows_failed: int,
    action_taken: str,
) -> None:
    
    import uuid
    _ensure_dq_results_table_exists(spark)
    row = spark.createDataFrame(
        [(str(uuid.uuid4()), batch_id, layer, table_name, check_name, severity,
          rows_checked, rows_failed, action_taken, datetime.utcnow())],
        schema=DQ_RESULTS_SCHEMA,
    )
    row.write.format("delta").mode("append").save(DQ_RESULTS_PATH)

In [0]:

def ensure_columns_present(df: DataFrame, expected_columns: dict) -> DataFrame:
    
    for col_name, col_type in expected_columns.items():
        if col_name not in df.columns:
            print(f"[schema_drift] Column '{col_name}' missing from source — adding as null ({col_type}).")
            df = df.withColumn(col_name, lit(None).cast(col_type))
    return df


In [0]:


def detect_unexpected_columns(df: DataFrame, expected_columns: set) -> list:

    return [c for c in df.columns if c not in expected_columns]


In [0]:
def flag_row_level_checks(df: DataFrame, checks: dict) -> DataFrame:

    failure_exprs = [when(condition, lit(name)) for name, condition in checks.items()]
    df = df.withColumn(
        "_dq_failures",
        sql_filter(array(*failure_exprs), lambda x: x.isNotNull())
    )
    return df